<h1 style=\"text-align: center; font-size: 50px;\"> Register Model </h1>

# Notebook Overview

- Start Execution
- Install and Import Libraries
- Configure Settings
- Register the Model Log Results to MLFlow

# Start Execution

In [1]:
import os
import logging
import time

# Configure logger
logger: logging.Logger = logging.getLogger("register_model_logger")
logger.setLevel(logging.INFO)
logger.propagate = False  # Prevent duplicate logs from parent loggers

# Set formatter
formatter: logging.Formatter = logging.Formatter(
    fmt="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

# Configure and attach stream handler
stream_handler: logging.StreamHandler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

In [2]:
start_time = time.time()  
logger.info("Notebook execution started.")

2026-02-04 17:37:25 - INFO - Notebook execution started.


# Install and Import Libraries

In [3]:
%%time

%pip install -r ../requirements.txt --quiet

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/lightning_utilities-0.14.0-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation.. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/deep_ep-1.0.0+a84a248-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation.. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/looseversion-1.3.0-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation.. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/opt_einsum-3.4.0-py3.12.egg is depreca

In [4]:
# ------------------------- Import Services -------------------------

import tempfile
import shutil
import warnings
import sys

import mlflow
import mlflow.pyfunc
from mlflow import MlflowClient
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec, TensorSpec, ParamSchema, ParamSpec
from mlflow.tracking import MlflowClient

# # Define the relative path to the 'src' directory (two levels up from current working directory)
src_path = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add 'src' directory to system path for module imports (e.g., utils)
if src_path not in sys.path:
    sys.path.append(src_path)

# Import new MLflow models-from-code components
from src.mlflow import Logger
from src.utils import (
    load_config,
)

src_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if src_path not in sys.path:
    sys.path.append(src_path)

# Correct import for onnx_utils
from src.onnx_utils import ModelExportConfig

# Configure Settings

In [5]:
# ------------------------ Suppress Verbose Logs ------------------------
warnings.filterwarnings("ignore")

# Suppress NeMo internal logging
logging.getLogger('nemo_logger').setLevel(logging.ERROR)

In [6]:
# ------------------------- Model File Paths -------------------------
MT_MODEL = "Helsinki-NLP/opus-mt-en-es"
ASR_MODEL_PATH = "/home/jovyan/datafabric/STT_En_Citrinet_1024_Gamma_0.25/stt_en_citrinet_1024_gamma_0_25.nemo"                  # Speech-to-Text (ASR) model
SPECTROGRAM_GENERATOR_PATH = "/home/jovyan/datafabric/TTS_Es_Multispeaker_FastPitch_HiFiGAN/tts_es_fastpitch_multispeaker.nemo"  # Spectrogram generator model (FastPitch)
VOCODER_PATH = "/home/jovyan/datafabric/TTS_Es_Multispeaker_FastPitch_HiFiGAN/tts_es_hifigan_ft_fastpitch_multispeaker.nemo"     # Vocoder model (HiFiGAN)
CONFIG_PATH = "../configs/config.yaml"
AUDIO_SAMPLE_PATH = "../data/ForrestGump.mp3"      # Path to the input English audio sample

# ------------------------- MLflow Experiment Configuration -------------------------

EXPERIMENT_NAME = "NeMo_Translation_Experiment"    # MLflow experiment name
RUN_NAME = "NeMo_en_es_Translation_Run"            # Specific run name inside the experiment
MODEL_NAME = "nemo_en_es"                          # Registered model name in MLflow
DEMO_PATH = "../demo"                              # Path to save demo outputs

In [7]:
# Load configuration and secrets
config = load_config(CONFIG_PATH)

print("✅ Configuration loaded successfully")

✅ Configuration loaded successfully


In [8]:
# ------------------------- Helper Functions -------------------------

def create_nemo_models_dict():
    """Create NeMo models dictionary from defined paths."""
    return {
        "enc_dec_CTC": ASR_MODEL_PATH,
        "fast_pitch": SPECTROGRAM_GENERATOR_PATH,
        "hifi_gan": VOCODER_PATH
    }

def load_models_for_onnx_conversion():
    """
    Load all models into memory for ONNX conversion.
    
    Returns:
        tuple: (nemo_models_dict, loaded_models_dict)
    """
    import torch
    import nemo.collections.asr as nemo_asr
    import nemo.collections.tts as nemo_tts
    from transformers import MarianMTModel, MarianTokenizer
    
    nemo_models = create_nemo_models_dict()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Load all models into memory
    loaded_models = {
        'mt_model': MarianMTModel.from_pretrained(MT_MODEL),
        'asr_model': nemo_asr.models.EncDecCTCModel.restore_from(nemo_models["enc_dec_CTC"]),
        'fast_pitch_model': nemo_tts.models.FastPitchModel.restore_from(nemo_models["fast_pitch"]),
        'hifi_gan_model': nemo_tts.models.HifiGanModel.restore_from(nemo_models["hifi_gan"])
    }
    
    logger.info("All models loaded into memory for ONNX conversion")
    return nemo_models, loaded_models

def create_and_convert_onnx_models(loaded_models):
    """
    Create ONNX versions of all models and save them to a temporary directory.
    
    Args:
        loaded_models: Dictionary of loaded model objects
    
    Returns:
        str: Path to directory containing ONNX model files
    """
    import torch
    from optimum.onnxruntime import ORTModelForSeq2SeqLM
    from optimum.exporters.onnx import main_export
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Create temp directory for ONNX models
    onnx_dir = os.path.join(tempfile.gettempdir(), "onnx_models")
    if os.path.exists(onnx_dir):
        shutil.rmtree(onnx_dir)
    os.makedirs(onnx_dir)
    
    try:
        # Convert Transformers model to ONNX
        mt_model = loaded_models['mt_model']
        mt_onnx_path = os.path.join(onnx_dir, "Helsinki-NLP.onnx")
        
        # Export using optimum
        dummy_input = {"input_ids": torch.randint(0, 1000, (1, 10))}
        torch.onnx.export(
            mt_model,
            dummy_input,
            mt_onnx_path,
            export_params=True,
            opset_version=12,
            do_constant_folding=True,
            input_names=['input_ids'],
            output_names=['output'],
            dynamic_axes={'input_ids': {0: 'batch_size', 1: 'sequence'},
                         'output': {0: 'batch_size', 1: 'sequence'}}
        )
        logger.info(f"Converted Helsinki-NLP to ONNX: {mt_onnx_path}")
        
        # Convert NeMo models to ONNX using their built-in export
        nemo_models_info = [
            ('asr_model', 'enc_dec_CTC.onnx'),
            ('fast_pitch_model', 'fast_pitch.onnx'), 
            ('hifi_gan_model', 'hifi_gan.onnx')
        ]
        
        for model_key, onnx_filename in nemo_models_info:
            model = loaded_models[model_key].to(device)
            onnx_path = os.path.join(onnx_dir, onnx_filename)
            
            # Use NeMo's built-in ONNX export
            model.export(onnx_path, check_trace=False)
            logger.info(f"Converted {model_key} to ONNX: {onnx_path}")
            
        logger.info(f"All models converted to ONNX in directory: {onnx_dir}")
        return onnx_dir
        
    except Exception as e:
        logger.error(f"Error during ONNX conversion: {str(e)}")
        logger.info("Continuing without ONNX conversion...")
        # Return empty directory if conversion fails
        return onnx_dir

def prepare_models_with_onnx(nemo_models, onnx_dir):
    """
    Prepare models directory containing both .nemo and .onnx files.
    
    Args:
        nemo_models: Dictionary of NeMo model paths
        onnx_dir: Directory containing ONNX model files
    
    Returns:
        str: Path to directory containing all model files
    """
    models_dir = os.path.join(tempfile.gettempdir(), "all_models")
    if os.path.exists(models_dir):
        shutil.rmtree(models_dir)
    os.makedirs(models_dir)
    
    # Copy NeMo models
    for model_name, model_path in nemo_models.items():
        if os.path.exists(model_path):
            target_filename = f"{model_name}.nemo"
            shutil.copy2(model_path, os.path.join(models_dir, target_filename))
            logger.info(f"Copied NeMo model: {model_name} -> {target_filename}")
    
    # Copy ONNX models if they exist
    if os.path.exists(onnx_dir):
        for onnx_file in os.listdir(onnx_dir):
            if onnx_file.endswith('.onnx'):
                src_path = os.path.join(onnx_dir, onnx_file)
                dst_path = os.path.join(models_dir, onnx_file)
                shutil.copy2(src_path, dst_path)
                logger.info(f"Copied ONNX model: {onnx_file}")
    
    return models_dir

def create_dummy_docs_directory():
    """Create minimal docs directory for vanilla-rag compatibility."""
    temp_docs_dir = os.path.join(tempfile.gettempdir(), "dummy_docs")
    if os.path.exists(temp_docs_dir):
        shutil.rmtree(temp_docs_dir)
    os.makedirs(temp_docs_dir)
    
    # Create a dummy file
    dummy_file = os.path.join(temp_docs_dir, "README.md")
    with open(dummy_file, 'w') as f:
        f.write("# NeMo Audio Translation Model\n\nThis model uses NeMo for audio translation with ONNX support.")
    
    return temp_docs_dir

## Register the Model Log Results to MLFlow

In [9]:
%%time

mlflow.set_tracking_uri('/phoenix/mlflow')

# Set the MLflow experiment name
mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

logger.info(f'Starting the experiment: {EXPERIMENT_NAME}')
logger.info(f"Using MLflow tracking URI: {mlflow.get_tracking_uri()}")

input_schema = Schema([
    ColSpec("string", "source_text"),
    ColSpec("string", "source_serialized_audio"),
])

output_schema = Schema([
ColSpec("string", "original_text"),
ColSpec("string", "translated_text"),
ColSpec("string", "translated_serialized_audio"),
])

params_schema = ParamSchema([
ParamSpec("use_audio", "boolean", False)
])

signature = ModelSignature(inputs=input_schema, outputs=output_schema, params=params_schema)


logger.info("Starting NeMo Audio Translation model registration with ONNX conversion...")

# Load all models and convert to ONNX
nemo_models, loaded_models = load_models_for_onnx_conversion()

# Convert models to ONNX format
onnx_dir = create_and_convert_onnx_models(loaded_models)

# Prepare combined models directory (both .nemo and .onnx)
models_dir = prepare_models_with_onnx(nemo_models, onnx_dir)
docs_dir = create_dummy_docs_directory()


with mlflow.start_run(run_name=RUN_NAME) as run:
    # Print the artifact URI for reference
    logging.info(f"Run's Artifact URI: {run.info.artifact_uri}")

    Logger.log_model(
        signature=signature,
        artifact_path=MODEL_NAME,
        config_path=CONFIG_PATH,
        docs_path=docs_dir,      # Dummy docs for interface compliance
        model_path=models_dir,   # Combined .nemo and .onnx models -> /artifacts/data/models/
        demo_folder=DEMO_PATH
    )
    mlflow.register_model(
        model_uri=f"runs:/{run.info.run_id}/{MODEL_NAME}",
        name=MODEL_NAME
    )
    
logger.info(f"✅ NeMo Audio Translation model '{MODEL_NAME}' registered successfully with ONNX support!")

2026/02/04 17:37:34 INFO mlflow.tracking.fluent: Experiment with name 'NeMo_Translation_Experiment' does not exist. Creating a new experiment.
2026-02-04 17:37:34 - INFO - Starting the experiment: NeMo_Translation_Experiment
2026-02-04 17:37:34 - INFO - Using MLflow tracking URI: /phoenix/mlflow
2026-02-04 17:37:34 - INFO - Starting NeMo Audio Translation model registration with ONNX conversion...


[NeMo I 2026-02-04 17:37:59 nemo_logging:393] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2026-02-04 17:37:59 nemo_logging:405] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: null
    sample_rate: 16000
    batch_size: 32
    trim_silence: false
    max_duration: 20.0
    shuffle: true
    is_tarred: false
    tarred_audio_filepaths: null
    use_start_end_token: false
    
[NeMo W 2026-02-04 17:37:59 nemo_logging:405] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath: null
    sample_rate: 16000
    batch_size: 32
    shuffle: false
    use_start_end_token: false
    
[NeMo W 2026-02-04 17:37:59 nemo_logging:405] Please call the ModelPT.setup_test_data() or ModelPT.setup_multiple_test_data() method 

[NeMo I 2026-02-04 17:37:59 nemo_logging:393] PADDING: 16
[NeMo I 2026-02-04 17:38:03 nemo_logging:393] Model EncDecCTCModelBPE was successfully restored from /home/jovyan/datafabric/STT_En_Citrinet_1024_Gamma_0.25/stt_en_citrinet_1024_gamma_0_25.nemo.


[NeMo W 2026-02-04 17:38:07 nemo_logging:405] Function ``g2p_backward_compatible_support`` is deprecated. But it will not be removed until a further notice. G2P object root directory `nemo_text_processing.g2p` has been replaced with `nemo.collections.tts.g2p`. Please use the latter instead as of NeMo 1.18.0.
[NeMo W 2026-02-04 17:38:07 nemo_logging:405] `<class 'nemo.collections.tts.g2p.models.i18n_ipa.IpaG2p'>` is experimental and not ready for production yet. Use at your own risk.
[NeMo W 2026-02-04 17:38:07 nemo_logging:405] apply_to_oov_word=None, This means that some of words will remain unchanged if they are not handled by any of the rules in self.parse_one_word(). This may be intended if phonemes and chars are both valid inputs, otherwise, you may see unexpected deletions in your input.
[NeMo W 2026-02-04 17:38:07 nemo_logging:405] `<class 'nemo.collections.common.tokenizers.text_to_speech.tts_tokenizers.IPATokenizer'>` is experimental and not ready for production yet. Use at yo

...

[NeMo I 2026-02-04 17:39:05 nemo_logging:393] Successfully exported FastPitchModel to /tmp/tmpynjzvend.onnx


2026-02-04 17:39:06 - INFO - Model saved to ONNX: fast_pitch/model.onnx
2026-02-04 17:39:06 - INFO - ✅ Converted fast_pitch to directory: fast_pitch
2026-02-04 17:39:06 - INFO - 🔄 Converting nemo model: hifi_gan
2026-02-04 17:39:06 - INFO - 📁 Model directory: hifi_gan
2026-02-04 17:39:06 - INFO - 🔍 Model identified as: nemo
2026-02-04 17:39:06 - INFO - 🔄 Exporting loaded NeMo model with format: ONNX
2026-02-04 17:39:06 - INFO - 🔄 Using official NVIDIA export() method


Removing weight norm...
Removing weight norm...
[NeMo I 2026-02-04 17:39:07 nemo_logging:393] Successfully exported HifiGanModel to /tmp/tmpo86p11o5.onnx


2026-02-04 17:39:08 - INFO - Model saved to ONNX: hifi_gan/model.onnx
2026-02-04 17:39:08 - INFO - ✅ Converted hifi_gan to directory: hifi_gan
2026-02-04 17:39:08 - INFO - 📦 Added model directory artifact: model_Helsinki-NLP -> Helsinki-NLP
2026-02-04 17:39:08 - INFO - 📦 Added model directory artifact: model_enc_dec_CTC -> enc_dec_CTC
2026-02-04 17:39:08 - INFO - 📦 Added model directory artifact: model_fast_pitch -> fast_pitch
2026-02-04 17:39:08 - INFO - 📦 Added model directory artifact: model_hifi_gan -> hifi_gan
2026-02-04 17:39:08 - INFO -   No Triton structure requested, using model directories as-is
2026-02-04 17:39:19 - INFO - Model logged with artifacts: ['model_Helsinki-NLP', 'model_enc_dec_CTC', 'model_fast_pitch', 'model_hifi_gan']
2026-02-04 17:39:19 - INFO - ✅ Model logged with 4 model directories created!
Successfully registered model 'nemo_en_es'.
2026/02/04 17:39:19 WARNING mlflow.tracking._model_registry.fluent: Run with id d7a4a91065c04eb489eff92a3ca75b91 has no artif

CPU times: user 56.5 s, sys: 19.7 s, total: 1min 16s
Wall time: 1min 45s


In [10]:
# ------------------------- Success Confirmation -------------------------

print(f"✅ Model '{MODEL_NAME}' successfully logged and registered under experiment '{EXPERIMENT_NAME}'.")

✅ Model 'nemo_en_es' successfully logged and registered under experiment 'NeMo_Translation_Experiment'.


In [11]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")
logger.info("✅ Notebook execution completed successfully.")

2026-02-04 17:39:19 - INFO - ⏱️ Total execution time: 1m 54.24s
2026-02-04 17:39:19 - INFO - ✅ Notebook execution completed successfully.


Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).